# Can we predict 30-day readmission (`readmit_30d`)?

**Plan**
1. Inspect the data (the same person appears in many rows, which matters for validation).
2. Evaluate models with **grouped** cross-validation (all visits of a person stay in the same fold), so we measure generalisation to *new people*.
3. Compare against a trivial baseline; look at discrimination (ROC-AUC, PR-AUC), calibration and feature importance.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width', 140)
df = pd.read_csv('data.csv'); cols = pd.read_csv('columns.csv')
print(df.shape); display(cols); display(df.head())

## 1. Data checks

In [ ]:
print('missing values:', df.isna().sum().sum(), '| duplicate rows:', df.duplicated().sum())
print('people:', df.person_key.nunique(), '| visits/person (min, median, max):', df.groupby('person_key').size().describe()[['min','50%','max']].tolist())
print('readmission rate: %.3f' % df.readmit_30d.mean())
for c in ['birth_year', 'sex', 'postcode_area']:
    print(c, 'constant within person:', (df.groupby('person_key')[c].nunique() == 1).all())
display(df.describe().T)

In [ ]:
# Is there a person effect on readmission? If so, row-wise random CV would be optimistic.
pr = df.groupby('person_key').readmit_30d.agg(['mean', 'size'])
p0 = df.readmit_30d.mean()
print('observed between-person SD of readmit rate: %.3f' % pr['mean'].std())
print('SD expected from chance alone at p=%.3f: %.3f' % (p0, np.sqrt(p0*(1-p0)/pr['size']).mean()))

## 2. Univariate look at the features

In [ ]:
feats = ['visit_number','birth_year','sex','postcode_area','bmi','length_of_stay_days','n_medications','lab_a','lab_b']
d = df.copy(); d['sex'] = (d.sex == 'M').astype(int)
fig, ax = plt.subplots(3, 3, figsize=(13, 10))
for a, c in zip(ax.ravel(), feats):
    if d[c].nunique() > 12:
        gg = d.groupby(pd.qcut(d[c], 10, duplicates='drop'), observed=True).readmit_30d.mean()
        labels = [f'{i.mid:.1f}' for i in gg.index]
    else:
        gg = d.groupby(c).readmit_30d.mean(); labels = list(gg.index)
    x = range(len(gg)); a.plot(x, gg.values, 'o-'); a.set_xticks(x); a.set_xticklabels(labels, rotation=45)
    a.axhline(p0, ls='--', c='grey'); a.set_title(c); a.set_ylabel('readmit rate')
plt.tight_layout(); plt.show()

## 3. Models with grouped cross-validation
`person_key` is used only as the CV group, never as a feature. Metrics are on out-of-fold predictions.

In [ ]:
X = d[feats]; y = d.readmit_30d.values; g = d.person_key.values
models = {
    'baseline (prevalence)': DummyClassifier(strategy='prior'),
    'logistic regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'gradient boosting': HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, l2_regularization=1.0, random_state=0),
}
cv = GroupKFold(n_splits=5)
oof = {}; rows = []
for n, m in models.items():
    p = cross_val_predict(m, X, y, groups=g, cv=cv, method='predict_proba')[:, 1]; oof[n] = p
    folds = [roc_auc_score(y[t], p[t]) for _, t in cv.split(X, y, g)] if not n.startswith('baseline') else [0.5]
    rows.append(dict(model=n, ROC_AUC=roc_auc_score(y, p), AUC_fold_sd=np.std(folds), PR_AUC=average_precision_score(y, p), Brier=brier_score_loss(y, p)))
res = pd.DataFrame(rows).set_index('model'); display(res.round(4))
print('PR-AUC of a random model = prevalence = %.3f' % y.mean())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for n, p in oof.items():
    if n.startswith('baseline'): continue
    fpr, tpr, _ = roc_curve(y, p); ax[0].plot(fpr, tpr, label=f'{n} ({roc_auc_score(y, p):.3f})')
    pp_, rc, _ = precision_recall_curve(y, p); ax[1].plot(rc, pp_, label=n)
    fp, mp = calibration_curve(y, p, n_bins=10, strategy='quantile'); ax[2].plot(mp, fp, 'o-', label=n)
ax[0].plot([0, 1], [0, 1], 'k--'); ax[0].set(title='ROC', xlabel='FPR', ylabel='TPR'); ax[0].legend()
ax[1].axhline(y.mean(), c='k', ls='--'); ax[1].set(title='Precision-recall', xlabel='recall', ylabel='precision'); ax[1].legend()
ax[2].plot([0, 1], [0, 1], 'k--'); ax[2].set(title='Calibration (out-of-fold)', xlabel='mean predicted', ylabel='observed rate'); ax[2].legend()
plt.tight_layout(); plt.show()

## 4. Is the signal real? Bootstrap CI (resampling people) and shuffled-label null

In [ ]:
rng = np.random.default_rng(0)
best = 'logistic regression'; p = oof[best]
ppl = np.unique(g); idx = {k: np.where(g == k)[0] for k in ppl}
bs = []
for _ in range(300):
    s = np.concatenate([idx[k] for k in rng.choice(ppl, len(ppl))]); bs.append(roc_auc_score(y[s], p[s]))
print('%s ROC-AUC %.3f, 95%% CI over people [%.3f, %.3f]' % (best, roc_auc_score(y, p), *np.percentile(bs, [2.5, 97.5])))
null = []
for _ in range(20):
    ys = rng.permutation(y)
    pp = cross_val_predict(models[best], X, ys, groups=g, cv=cv, method='predict_proba')[:, 1]; null.append(roc_auc_score(ys, pp))
print('null AUC with shuffled labels: mean %.3f, max %.3f' % (np.mean(null), np.max(null)))

## 5. What drives the prediction?

In [ ]:
lr = models['logistic regression'].fit(X, y)
coef = pd.Series(lr[-1].coef_[0], index=feats).sort_values()
print('Standardised logistic coefficients (log-odds per 1 SD):'); display(coef.round(3))
imp = np.zeros(len(feats))
for tr, te in cv.split(X, y, g):
    m = models['gradient boosting'].fit(X.iloc[tr], y[tr])
    imp += permutation_importance(m, X.iloc[te], y[te], scoring='roc_auc', n_repeats=5, random_state=0).importances_mean / 5
pd.Series(imp, index=feats).sort_values().plot.barh(title='GBM permutation importance (drop in held-out AUC)'); plt.show()

## 6. Practical usefulness: risk stratification

In [ ]:
p = oof['logistic regression']
t = pd.DataFrame({'p': p, 'y': y}); t['decile'] = pd.qcut(t.p.rank(method='first'), 10, labels=False) + 1
display(t.groupby('decile').agg(mean_pred=('p', 'mean'), observed=('y', 'mean'), n=('y', 'size')).round(3))
top = t[t.decile == 10].y.mean()
print('Top-decile readmit rate %.1f%% vs overall %.1f%% (lift %.2fx)' % (100*top, 100*y.mean(), top/y.mean()))

## 7. Does a person's history help? (prior-visit readmissions)
Section 1 showed readmission clusters within people far beyond chance. For a *returning* patient, outcomes of their **earlier** visits are known at prediction time, so we can use them (strictly past information only: visits with a lower `visit_number`). Caveat: this only applies to people with prior visits, and assumes `visit_number` reflects chronological order.

In [ ]:
h = d.sort_values(['person_key', 'visit_number']).copy()
grp = h.groupby('person_key').readmit_30d
h['prior_n'] = grp.cumcount()
h['prior_readmits'] = grp.cumsum() - h.readmit_30d
h['prior_rate'] = (h.prior_readmits + p0 * 2) / (h.prior_n + 2)   # shrunk toward the base rate
feats2 = feats + ['prior_n', 'prior_rate']
Xh, yh, gh = h[feats2], h.readmit_30d.values, h.person_key.values
ph = cross_val_predict(models['logistic regression'], Xh, yh, groups=gh, cv=cv, method='predict_proba')[:, 1]
pb = cross_val_predict(models['logistic regression'], h[feats], yh, groups=gh, cv=cv, method='predict_proba')[:, 1]
print('Logistic AUC without history: %.3f | with prior-readmission history: %.3f' % (roc_auc_score(yh, pb), roc_auc_score(yh, ph)))
print('PR-AUC without: %.3f | with: %.3f' % (average_precision_score(yh, pb), average_precision_score(yh, ph)))
m = h.prior_n > 0
print('Restricted to visits with >=1 prior visit: without %.3f | with %.3f' % (roc_auc_score(yh[m], pb[m]), roc_auc_score(yh[m], ph[m])))

## Conclusions
- **Yes, partly.** On unseen people (grouped 5-fold CV), a plain logistic regression on admission/stay features reaches ROC-AUC about 0.65 (bootstrap 95% CI about 0.63 to 0.66) and PR-AUC about 0.32 against a 0.198 base rate. Shuffled labels give about 0.50. It is well calibrated, and the top risk decile is readmitted about 39% of the time (about 2x lift). That is a real but modest signal.
- **Drivers:** longer stay, more medications at discharge and higher `lab_a` raise risk. Demographics, BMI, `lab_b`, postcode and visit number add almost nothing. Gradient boosting does no better than logistic regression, so the signal is roughly linear and there are no strong interactions.
- **Readmission history is the strongest predictor.** Readmissions cluster within people well beyond chance (between-person SD 0.27 vs 0.15 expected). Adding earlier visits' readmission outcomes lifts AUC to about 0.82 (about 0.84 for returning patients). This only works for patients with prior visits and assumes `visit_number` is chronological and prior outcomes are available when predicting.
- **Validation caveat:** rows from one person are not independent, so a random row-wise split would be optimistic. All estimates here are grouped by `person_key`.
- **Limits:** no dates, diagnoses or discharge destination are in the data, and these results are associations, not causal. Before deployment, check the model on a later time period and check fairness across `sex` and `postcode_area`.